# COVER-KBC — Post-Architecture Real-Model Runtime Smoke

Runtime compatibility only: no benchmark split, no accuracy, no
TRAIN/VAL/TEST. Staged residency keeps one checkpoint on the GPU at a
time, so the same run works on ~24 GB and on A100.

Set `REPO_SHA`, then Runtime ▸ Run all. If memory is insufficient the
smoke fails clearly — it never falls back to a smaller model.


In [ ]:
# COVER-KBC — post-architecture real-model smoke: setup
REPO_URL  = "https://github.com/vquclinh/FactElicit-AKBC.git"
REPO_SHA  = ""                       # <- set to the commit you want to smoke
REPO_ROOT = "/content/FactElicit-AKBC"
CACHE_ROOT = "/content/hf-cache"
OUTPUT_ROOT = "/content/smoke-out"

import os, subprocess, sys, pathlib
os.makedirs(CACHE_ROOT, exist_ok=True); os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.environ["HF_HOME"] = CACHE_ROOT
# Sequential residency is the real fix; this only reduces fragmentation.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if not pathlib.Path(REPO_ROOT).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)
os.chdir(REPO_ROOT)
subprocess.run(["git", "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "checkout", "--force", REPO_SHA or "HEAD"], check=True)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", ".[hf]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "bitsandbytes", "mistral-common"], check=True)
print("ready:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                               text=True, check=True).stdout.strip())


In [ ]:
# Run the smoke. Staged residency: one checkpoint on the GPU at a time.
import subprocess, sys, json, pathlib

OUT = pathlib.Path(OUTPUT_ROOT) / "real_model_architecture_smoke_summary.json"
subprocess.run([sys.executable, "scripts/real_model_smoke.py",
                "--config", "configs/experiments/cover_kbc_v2_mistral24_qwen4.yaml",
                "--out", str(OUT)])

summary = json.loads(OUT.read_text())
print(json.dumps(summary, indent=2)[:4000])
print("\nRESULT:", summary["result"], "->", OUT)
